<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/02_preprocessing_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Prétraitement des données pour la traduction automatique

Dans ce notebook, nous préparons les données nettoyées afin de les rendre exploitables par un modèle de traduction automatique.

Après l’exploration du dataset dans le notebook précédent, nous disposons maintenant de trois fichiers :

- `train.csv` : données d'entraînement ;
- `valid.csv` : données de validation ;
- `test.csv` : données de test.

L’objectif de cette étape est de transformer les phrases en séquences numériques.  
Un modèle de Deep Learning ne comprend pas directement le texte brut : il faut donc convertir chaque mot en entier, puis uniformiser la longueur des phrases grâce au padding.

Nous allons préparer les données dans les deux sens :

- français → éwé ;
- éwé → français.

Cela nous permettra plus tard de construire un système de traduction bidirectionnel.

In [2]:
# On utilise l'outil d'upload de Google Colab pour charger les fichiers CSV préparés
# lors de l'étape précédente.
#
# Les fichiers attendus sont :
# - train.csv
# - valid.csv
# - test.csv
#
# Ces fichiers contiennent déjà les phrases français-éwé nettoyées et séparées
# en trois sous-ensembles.

from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving valid.csv to valid.csv


In [3]:
# On importe les bibliothèques nécessaires pour cette étape.
#
# pandas permet de manipuler les fichiers CSV.
# numpy est utile pour gérer les tableaux numériques.
# re permet d'appliquer quelques nettoyages simples sur les textes.
# TensorFlow/Keras sera utilisé ici pour la tokenisation et le padding.

import pandas as pd
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
# On charge les trois fichiers préparés dans le notebook précédent.
# Chaque fichier contient normalement les colonnes :
# french, ewe, source, type, french_length et ewe_length.

train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("valid.csv")
test_df = pd.read_csv("test.csv")

print("Train      :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test       :", test_df.shape)

Train      : (18820, 6)
Validation : (2352, 6)
Test       : (2353, 6)


In [5]:
# On affiche quelques lignes du dataset d'entraînement.
# Cela permet de vérifier que les colonnes sont bien présentes
# et que les phrases sont correctement alignées.

train_df.head()

,french,ewe,source,type,french_length,ewe_length
0,"""C'est moins lourd pour les enfants, pour le s...",ele hodzoe na ɖeviwo kple ame bubu geɖewo,https://ellecitoyenne.com/,"Blog, News",14,8
1,"""Il ne nous reste que nous, si nous voulons un...",mía ŋutɔwo dzi ɖeɖe ko wole be míaka ɖo ne m...,https://ellecitoyenne.com/,"Blog, News",15,13
2,"""Pendant quelques années, Histoire et History ...",tsakakatsaka ɖeke magava eme le akɔdada le kam...,https://ellecitoyenne.com/,"Blog, News",47,14
3,"""Ainsi, à travers une lecture littérale de la ...",Aleae to bibla xexle me la miedoa dzesi be duk...,https://ellecitoyenne.com/,"Blog, News",21,21
4,"""Le prof d'EPS qui donne des cours de langue ?...",meli ʋliʋlim xena fetu sɔsɔe xɔxɔ le dɔ ƒomevi...,https://ellecitoyenne.com/,"Blog, News",45,11


In [6]:

print("Colonnes du dataset :")
print(train_df.columns.tolist())

Colonnes du dataset :
['french', 'ewe', 'source', 'type', 'french_length', 'ewe_length']


## Nettoyage léger des textes

Pour la traduction automatique, il faut nettoyer les phrases sans détruire l'information linguistique.

Nous allons donc :

- convertir le texte en minuscules ;
- supprimer les espaces inutiles ;
- séparer légèrement certains signes de ponctuation ;
- éviter de supprimer les accents ou les caractères propres à l’éwé.

Un nettoyage trop agressif pourrait rendre les phrases moins naturelles et nuire à l’apprentissage du modèle.

In [7]:
# Cette fonction applique un nettoyage simple et prudent.
#
# On ne supprime pas les accents ni les caractères propres à l'éwé,
# car ils peuvent être importants pour le sens des mots.
#
# L'objectif est seulement de rendre les phrases un peu plus homogènes :
# - passage en minuscules ;
# - suppression des espaces en trop ;
# - séparation de quelques signes de ponctuation.

def clean_text(text):
    text = str(text).lower()

    # On ajoute des espaces autour de certains signes de ponctuation.
    # Cela aide le tokenizer à mieux les repérer.
    text = re.sub(r"([?.!,¿])", r" \1 ", text)

    # On remplace plusieurs espaces successifs par un seul espace.
    text = re.sub(r"\s+", " ", text)

    # On enlève les espaces au début et à la fin de la phrase.
    text = text.strip()

    return text

In [8]:
# On applique le nettoyage aux colonnes françaises et éwé
# dans les trois ensembles : train, validation et test.
#
# On crée de nouvelles colonnes pour garder les textes originaux disponibles.

train_df["french_clean"] = train_df["french"].apply(clean_text)
train_df["ewe_clean"] = train_df["ewe"].apply(clean_text)

valid_df["french_clean"] = valid_df["french"].apply(clean_text)
valid_df["ewe_clean"] = valid_df["ewe"].apply(clean_text)

test_df["french_clean"] = test_df["french"].apply(clean_text)
test_df["ewe_clean"] = test_df["ewe"].apply(clean_text)

In [9]:

for i in range(5):
    print("Exemple", i + 1)
    print("FR original :", train_df.iloc[i]["french"])
    print("FR nettoyé  :", train_df.iloc[i]["french_clean"])
    print("EWE original:", train_df.iloc[i]["ewe"])
    print("EWE nettoyé :", train_df.iloc[i]["ewe_clean"])
    print("-" * 80)

Exemple 1
FR original : "C'est moins lourd pour les enfants, pour le systeme et beaucoup plus effectif. "
FR nettoyé  : "c'est moins lourd pour les enfants , pour le systeme et beaucoup plus effectif . "
EWE original: ele hodzoe na ɖeviwo kple ame bubu geɖewo
EWE nettoyé : ele hodzoe na ɖeviwo kple ame bubu geɖewo
--------------------------------------------------------------------------------
Exemple 2
FR original : "Il ne nous reste que nous, si nous voulons un jour espérer un changement. "
FR nettoyé  : "il ne nous reste que nous , si nous voulons un jour espérer un changement . "
EWE original: mía ŋutɔwo dzi ɖeɖe ko wole be míaka ɖo ne miedzi tɔtrɔ la
EWE nettoyé : mía ŋutɔwo dzi ɖeɖe ko wole be míaka ɖo ne miedzi tɔtrɔ la
--------------------------------------------------------------------------------
Exemple 3
FR original : "Pendant quelques années, Histoire et History étaient également enseignées dans l'autre langue, jusqu'à ce qu'un beau matin les Inspecteurs Pédagogiques N

## Ajout des tokens de début et de fin

Dans un modèle Seq2Seq, le décodeur doit apprendre à générer une phrase mot par mot.

Pour cela, on ajoute deux tokens spéciaux :

- `<start>` : indique le début de la phrase à générer ;
- `<end>` : indique la fin de la phrase.

Ces tokens seront utilisés plus tard pour entraîner le modèle à produire correctement une traduction complète.

In [10]:
# Cette fonction ajoute un token de début et un token de fin à une phrase.
# Ces marqueurs seront surtout utiles pour la partie cible du modèle Seq2Seq.

def add_start_end_tokens(text):
    return "<start> " + str(text) + " <end>"

In [11]:
# Pour la traduction français → éwé :
# - la source est la phrase française ;
# - la cible est la phrase en éwé.
#
# On ajoute donc les tokens <start> et <end> aux phrases en éwé,
# car ce sont elles que le modèle devra générer.

train_df["ewe_target"] = train_df["ewe_clean"].apply(add_start_end_tokens)
valid_df["ewe_target"] = valid_df["ewe_clean"].apply(add_start_end_tokens)
test_df["ewe_target"] = test_df["ewe_clean"].apply(add_start_end_tokens)

In [12]:
# Pour la traduction éwé → français :
# - la source est la phrase en éwé ;
# - la cible est la phrase française.
#
# Cette fois, on ajoute les tokens <start> et <end> aux phrases françaises.

train_df["french_target"] = train_df["french_clean"].apply(add_start_end_tokens)
valid_df["french_target"] = valid_df["french_clean"].apply(add_start_end_tokens)
test_df["french_target"] = test_df["french_clean"].apply(add_start_end_tokens)

In [13]:


print("Cible éwé avec tokens :")
print(train_df.iloc[0]["ewe_target"])

print("\nCible française avec tokens :")
print(train_df.iloc[0]["french_target"])

Cible éwé avec tokens :
<start> ele hodzoe na ɖeviwo kple ame bubu geɖewo <end>

Cible française avec tokens :
<start> "c'est moins lourd pour les enfants , pour le systeme et beaucoup plus effectif . " <end>


In [14]:
# On définit une taille maximale de vocabulaire.
# Cela permet de limiter le nombre de mots différents gardés par le tokenizer.
#
# Les mots trop rares seront remplacés par le token <unk>,
# qui représente les mots inconnus.

VOCAB_SIZE = 10000

In [15]:
# Tokenizer français.
# Il sera utilisé pour convertir les phrases françaises en séquences numériques.

french_tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    filters='',
    oov_token="<unk>"
)

# Tokenizer éwé.
# Il sera utilisé pour convertir les phrases éwé en séquences numériques.

ewe_tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    filters='',
    oov_token="<unk>"
)

In [16]:

# on entraîne les tokenizers uniquement sur train_df.
#
# On évite d'utiliser valid_df ou test_df ici,
# car ces ensembles doivent rester des données "jamais vues"
# pour évaluer correctement le modèle.

french_tokenizer.fit_on_texts(
    train_df["french_clean"].tolist() + train_df["french_target"].tolist()
)

ewe_tokenizer.fit_on_texts(
    train_df["ewe_clean"].tolist() + train_df["ewe_target"].tolist()
)

In [17]:
# On récupère la taille réelle des vocabulaires.
# On ajoute +1 car l'indice 0 est réservé au padding.

french_vocab_size = min(VOCAB_SIZE, len(french_tokenizer.word_index) + 1)
ewe_vocab_size = min(VOCAB_SIZE, len(ewe_tokenizer.word_index) + 1)

print("Taille du vocabulaire français :", french_vocab_size)
print("Taille du vocabulaire éwé      :", ewe_vocab_size)

Taille du vocabulaire français : 10000
Taille du vocabulaire éwé      : 10000


In [18]:
# On affiche quelques mots du vocabulaire français.
list(french_tokenizer.word_index.items())[:20]

[('<unk>', 1),
 (',', 2),
 ('de', 3),
 ('<start>', 4),
 ('<end>', 5),
 ('la', 6),
 ('et', 7),
 ('à', 8),
 ('le', 9),
 ('les', 10),
 ('que', 11),
 ('.', 12),
 ('des', 13),
 ('en', 14),
 ('pour', 15),
 ('je', 16),
 ('un', 17),
 ('pas', 18),
 ('qui', 19),
 ('a', 20)]

In [19]:
# On affiche quelques mots du vocabulaire éwé.

list(ewe_tokenizer.word_index.items())[:20]

[('<unk>', 1),
 ('<start>', 2),
 ('<end>', 3),
 ('le', 4),
 ('la', 5),
 ('be', 6),
 (',', 7),
 ('o', 8),
 ('me', 9),
 ('ɖe', 10),
 ('kple', 11),
 ('si', 12),
 ('na', 13),
 ('nye', 14),
 ('eye', 15),
 ('ƒe', 16),
 ('nu', 17),
 ('ŋu', 18),
 ('dzi', 19),
 ('ame', 20)]

In [20]:
# On calcule les longueurs des phrases nettoyées.
# Cela nous aide à choisir une longueur maximale adaptée pour le padding.

train_df["french_clean_length"] = train_df["french_clean"].apply(lambda x: len(x.split()))
train_df["ewe_clean_length"] = train_df["ewe_clean"].apply(lambda x: len(x.split()))

print("Statistiques des longueurs françaises :")
print(train_df["french_clean_length"].describe())

print("\nStatistiques des longueurs éwé :")
print(train_df["ewe_clean_length"].describe())

Statistiques des longueurs françaises :
count    18820.000000
mean        18.574655
std         11.256994
min          1.000000
25%         10.000000
50%         17.000000
75%         24.000000
max        103.000000
Name: french_clean_length, dtype: float64

Statistiques des longueurs éwé :
count    18820.000000
mean        12.607386
std          6.770301
min          1.000000
25%          8.000000
50%         12.000000
75%         16.000000
max         77.000000
Name: ewe_clean_length, dtype: float64


In [21]:
# Les percentiles permettent de savoir quelle longueur couvre la majorité des phrases.
# Par exemple, le percentile 95 indique une longueur qui couvre environ 95% des phrases.

print("95e percentile français :", np.percentile(train_df["french_clean_length"], 95))
print("95e percentile éwé      :", np.percentile(train_df["ewe_clean_length"], 95))

print("99e percentile français :", np.percentile(train_df["french_clean_length"], 99))
print("99e percentile éwé      :", np.percentile(train_df["ewe_clean_length"], 99))

95e percentile français : 39.0
95e percentile éwé      : 25.0
99e percentile français : 56.0
99e percentile éwé      : 33.0


In [22]:
# Pour une première version du modèle, on choisit une longueur maximale de 50 mots.

MAX_LEN_FRENCH = 50
MAX_LEN_EWE = 50

In [23]:
# Pour la direction français → éwé :
# l'entrée du modèle sera la phrase française,
# et la cible sera la phrase éwé avec les tokens <start> et <end>.

X_train_fr = french_tokenizer.texts_to_sequences(train_df["french_clean"])
y_train_ewe = ewe_tokenizer.texts_to_sequences(train_df["ewe_target"])

X_valid_fr = french_tokenizer.texts_to_sequences(valid_df["french_clean"])
y_valid_ewe = ewe_tokenizer.texts_to_sequences(valid_df["ewe_target"])

X_test_fr = french_tokenizer.texts_to_sequences(test_df["french_clean"])
y_test_ewe = ewe_tokenizer.texts_to_sequences(test_df["ewe_target"])

In [24]:
# On applique le padding pour que toutes les phrases aient la même longueur.
#
# padding="post" signifie que les zéros sont ajoutés à la fin de la phrase.
# truncating="post" signifie que les phrases trop longues sont coupées à la fin.

X_train_fr_pad = pad_sequences(
    X_train_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

y_train_ewe_pad = pad_sequences(
    y_train_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

X_valid_fr_pad = pad_sequences(
    X_valid_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

y_valid_ewe_pad = pad_sequences(
    y_valid_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

X_test_fr_pad = pad_sequences(
    X_test_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

y_test_ewe_pad = pad_sequences(
    y_test_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

In [25]:
# On vérifie les dimensions des tableaux obtenus.
# Chaque ligne correspond à une phrase transformée en suite de nombres.

print("X_train_fr_pad :", X_train_fr_pad.shape)
print("y_train_ewe_pad:", y_train_ewe_pad.shape)

print("X_valid_fr_pad :", X_valid_fr_pad.shape)
print("y_valid_ewe_pad:", y_valid_ewe_pad.shape)

print("X_test_fr_pad  :", X_test_fr_pad.shape)
print("y_test_ewe_pad :", y_test_ewe_pad.shape)

X_train_fr_pad : (18820, 50)
y_train_ewe_pad: (18820, 50)
X_valid_fr_pad : (2352, 50)
y_valid_ewe_pad: (2352, 50)
X_test_fr_pad  : (2353, 50)
y_test_ewe_pad : (2353, 50)


In [26]:
# Pour la direction éwé → français :
# l'entrée du modèle sera la phrase en éwé,
# et la cible sera la phrase française avec les tokens <start> et <end>.

X_train_ewe = ewe_tokenizer.texts_to_sequences(train_df["ewe_clean"])
y_train_fr = french_tokenizer.texts_to_sequences(train_df["french_target"])

X_valid_ewe = ewe_tokenizer.texts_to_sequences(valid_df["ewe_clean"])
y_valid_fr = french_tokenizer.texts_to_sequences(valid_df["french_target"])

X_test_ewe = ewe_tokenizer.texts_to_sequences(test_df["ewe_clean"])
y_test_fr = french_tokenizer.texts_to_sequences(test_df["french_target"])

In [27]:
# On applique le padding pour obtenir des tableaux utilisables par un modèle.

X_train_ewe_pad = pad_sequences(
    X_train_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

y_train_fr_pad = pad_sequences(
    y_train_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

X_valid_ewe_pad = pad_sequences(
    X_valid_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

y_valid_fr_pad = pad_sequences(
    y_valid_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

X_test_ewe_pad = pad_sequences(
    X_test_ewe,
    maxlen=MAX_LEN_EWE,
    padding="post",
    truncating="post"
)

y_test_fr_pad = pad_sequences(
    y_test_fr,
    maxlen=MAX_LEN_FRENCH,
    padding="post",
    truncating="post"
)

In [28]:
# On vérifie les dimensions pour la direction éwé → français.

print("X_train_ewe_pad :", X_train_ewe_pad.shape)
print("y_train_fr_pad  :", y_train_fr_pad.shape)

print("X_valid_ewe_pad :", X_valid_ewe_pad.shape)
print("y_valid_fr_pad  :", y_valid_fr_pad.shape)

print("X_test_ewe_pad  :", X_test_ewe_pad.shape)
print("y_test_fr_pad   :", y_test_fr_pad.shape)

X_train_ewe_pad : (18820, 50)
y_train_fr_pad  : (18820, 50)
X_valid_ewe_pad : (2352, 50)
y_valid_fr_pad  : (2352, 50)
X_test_ewe_pad  : (2353, 50)
y_test_fr_pad   : (2353, 50)


In [29]:
# Dans un modèle Seq2Seq, le décodeur reçoit une phrase décalée.
#
# decoder_input contient tous les mots sauf le dernier.
# decoder_target contient tous les mots sauf le premier.
#
# Cela apprend au modèle à prédire le mot suivant à chaque étape.

def create_decoder_inputs_targets(target_sequences):
    decoder_input = target_sequences[:, :-1]
    decoder_target = target_sequences[:, 1:]
    return decoder_input, decoder_target

In [30]:
# Préparation des entrées et sorties du décodeur pour le modèle français → éwé.

decoder_input_train_ewe, decoder_target_train_ewe = create_decoder_inputs_targets(y_train_ewe_pad)
decoder_input_valid_ewe, decoder_target_valid_ewe = create_decoder_inputs_targets(y_valid_ewe_pad)
decoder_input_test_ewe, decoder_target_test_ewe = create_decoder_inputs_targets(y_test_ewe_pad)

print("Decoder input train éwé :", decoder_input_train_ewe.shape)
print("Decoder target train éwé:", decoder_target_train_ewe.shape)

Decoder input train éwé : (18820, 49)
Decoder target train éwé: (18820, 49)


In [31]:
# Préparation des entrées et sorties du décodeur pour le modèle éwé → français.

decoder_input_train_fr, decoder_target_train_fr = create_decoder_inputs_targets(y_train_fr_pad)
decoder_input_valid_fr, decoder_target_valid_fr = create_decoder_inputs_targets(y_valid_fr_pad)
decoder_input_test_fr, decoder_target_test_fr = create_decoder_inputs_targets(y_test_fr_pad)

print("Decoder input train français :", decoder_input_train_fr.shape)
print("Decoder target train français:", decoder_target_train_fr.shape)

Decoder input train français : (18820, 49)
Decoder target train français: (18820, 49)


In [32]:
# On sauvegarde les tokenizers avec pickle.
# Cela nous évitera de les recréer à chaque fois.
#
# Le tokenizer français servira à transformer les phrases françaises en nombres.
# Le tokenizer éwé servira à transformer les phrases éwé en nombres.

import pickle

with open("french_tokenizer.pkl", "wb") as f:
    pickle.dump(french_tokenizer, f)

with open("ewe_tokenizer.pkl", "wb") as f:
    pickle.dump(ewe_tokenizer, f)

In [33]:
# On sauvegarde les données prétraitées au format NumPy.
# Ces fichiers seront utilisés directement dans le notebook d'entraînement du modèle.

np.save("X_train_fr_pad.npy", X_train_fr_pad)
np.save("decoder_input_train_ewe.npy", decoder_input_train_ewe)
np.save("decoder_target_train_ewe.npy", decoder_target_train_ewe)

np.save("X_valid_fr_pad.npy", X_valid_fr_pad)
np.save("decoder_input_valid_ewe.npy", decoder_input_valid_ewe)
np.save("decoder_target_valid_ewe.npy", decoder_target_valid_ewe)

np.save("X_test_fr_pad.npy", X_test_fr_pad)
np.save("decoder_input_test_ewe.npy", decoder_input_test_ewe)
np.save("decoder_target_test_ewe.npy", decoder_target_test_ewe)

In [34]:
# On sauvegarde aussi les données préparées pour la direction éwé → français.

np.save("X_train_ewe_pad.npy", X_train_ewe_pad)
np.save("decoder_input_train_fr.npy", decoder_input_train_fr)
np.save("decoder_target_train_fr.npy", decoder_target_train_fr)

np.save("X_valid_ewe_pad.npy", X_valid_ewe_pad)
np.save("decoder_input_valid_fr.npy", decoder_input_valid_fr)
np.save("decoder_target_valid_fr.npy", decoder_target_valid_fr)

np.save("X_test_ewe_pad.npy", X_test_ewe_pad)
np.save("decoder_input_test_fr.npy", decoder_input_test_fr)
np.save("decoder_target_test_fr.npy", decoder_target_test_fr)

In [35]:
# On sauvegarde quelques paramètres importants dans un dictionnaire.
# Ces valeurs seront réutilisées au moment de construire le modèle.

config = {
    "VOCAB_SIZE": VOCAB_SIZE,
    "french_vocab_size": french_vocab_size,
    "ewe_vocab_size": ewe_vocab_size,
    "MAX_LEN_FRENCH": MAX_LEN_FRENCH,
    "MAX_LEN_EWE": MAX_LEN_EWE
}

with open("preprocessing_config.pkl", "wb") as f:
    pickle.dump(config, f)

config

{'VOCAB_SIZE': 10000,
 'french_vocab_size': 10000,
 'ewe_vocab_size': 10000,
 'MAX_LEN_FRENCH': 50,
 'MAX_LEN_EWE': 50}

In [36]:
# On regroupe les fichiers importants dans une liste pour les télécharger.
# Ces fichiers pourront ensuite être placés dans le repo GitHub,
# plus précisément dans data/processed ou dans un dossier artifacts.

files_to_download = [
    "french_tokenizer.pkl",
    "ewe_tokenizer.pkl",
    "preprocessing_config.pkl",

    "X_train_fr_pad.npy",
    "decoder_input_train_ewe.npy",
    "decoder_target_train_ewe.npy",
    "X_valid_fr_pad.npy",
    "decoder_input_valid_ewe.npy",
    "decoder_target_valid_ewe.npy",
    "X_test_fr_pad.npy",
    "decoder_input_test_ewe.npy",
    "decoder_target_test_ewe.npy",

    "X_train_ewe_pad.npy",
    "decoder_input_train_fr.npy",
    "decoder_target_train_fr.npy",
    "X_valid_ewe_pad.npy",
    "decoder_input_valid_fr.npy",
    "decoder_target_valid_fr.npy",
    "X_test_ewe_pad.npy",
    "decoder_input_test_fr.npy",
    "decoder_target_test_fr.npy"
]

for file in files_to_download:
    files.download(file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

Dans ce notebook, nous avons préparé les données pour l'entraînement d'un modèle de traduction automatique français-éwé.

Nous avons commencé par charger les fichiers `train`, `validation` et `test`, puis nous avons appliqué un nettoyage léger aux textes. Ensuite, nous avons ajouté des tokens spéciaux de début et de fin pour préparer les phrases cibles au fonctionnement d’un modèle Seq2Seq.

Les phrases ont ensuite été transformées en séquences numériques grâce à des tokenizers, puis uniformisées avec du padding.

Enfin, nous avons préparé les données dans les deux directions :

- français → éwé ;
- éwé → français.

Ces fichiers seront utilisés dans le prochain notebook pour entraîner un premier modèle de traduction automatique basé sur une architecture Seq2Seq avec LSTM.